# unit10 レッスン: キャップストーン — アンサンブルと推論の運用

**題材** — Day2〜Day4 で育てた**あのコンペ**(中古品マーケットプレイスの価格予測)に戻る。
評価は RMSLE(= log 空間の RMSE)、検証は Day2 で確定した `GroupKFold(5)` を `product_key` で切る。

## このレッスンを終えると作れるようになるもの

1. 複数モデルの **OOF 予測をブレンド**し、重みを**手元のデータだけで**決められる
2. ブレンドが効かないとき、**なぜ効かないのかを数字で説明**できる(そして無駄に時間を溶かさない)
3. **シード平均**が効く条件と効かない条件を区別できる
4. **学習と推論を artifact で分離**し、別プロセスで同じ予測を再現できる
5. **GPU学習 + CPU推論 と LLM API の損益分岐**を自分で計算し、根拠をもって構成を選べる

所要の目安: **90〜120分**。このあと演習 `ex01`〜`ex04` が続く。

> 今日は10日間の締めくくりだ。新しいモデルは出てこない。**すでに持っている部品を組み上げて、
> 本番に載る形にする**のが仕事になる。

In [ ]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# Day2〜Day4 と同じコンペのデータを使う。unit10 直下でもリポジトリ直下でも動く。
DATA = Path("../unit02-validation-and-leakage/data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit02-validation-and-leakage/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

_UNIT_DIR = DATA.resolve().parent
ANSWER = _UNIT_DIR.parent / ".solutions" / _UNIT_DIR.name / "_answer.csv"
OUT = Path("output"); OUT.mkdir(exist_ok=True)

print("sklearn:", sklearn.__version__, "/ lightgbm:", lgb.__version__)
print("DATA =", DATA.resolve())

train = pd.read_csv(DATA / "train.csv", parse_dates=["collected_at"])
test = pd.read_csv(DATA / "test.csv", parse_dates=["collected_at"])
y = np.log1p(train["price"].to_numpy())
groups = train["product_key"]
gkf = GroupKFold(5)

NUM = ["brand_tier", "views", "title_len"]
CAT = ["category", "site", "condition"]
ORIGIN = pd.Timestamp("2025-10-01")

def featurize(df):
    X = df[NUM + CAT].copy()
    X["days"] = (df["collected_at"] - ORIGIN).dt.days
    return X

X_train, X_test = featurize(train), featurize(test)
print("X_train:", X_train.shape, " X_test:", X_test.shape)


def rmse_log(a, b):
    """log 空間の RMSE(= RMSLE)。小さいほど良い。"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def ridge_model():
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM + ["days"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])
    return make_pipeline(pre, Ridge(alpha=1.0))


def lgbm_model(seed=0, bagging=True):
    # n_jobs=1: Jupyter カーネル内で LightGBM の既定スレッド(全コア使用)が
    # カーネルの内部スレッドと衝突してデッドロックすることがあるため固定する。
    kw = dict(n_estimators=500, learning_rate=0.05, random_state=seed, verbose=-1, n_jobs=1)
    if bagging:
        kw.update(subsample=0.8, subsample_freq=1, colsample_bytree=0.8)
    return lgb.LGBMRegressor(**kw)


def make_oof(make, as_category):
    """GroupKFold(product_key) で OOF 予測と test 予測(fold平均)を作る。Day2 で確定した検証方法。"""
    oof = np.zeros(len(y)); pred_test = np.zeros(len(X_test))
    for tr_idx, va_idx in gkf.split(X_train, y, groups):
        Xa, Xb, Xq = X_train.iloc[tr_idx].copy(), X_train.iloc[va_idx].copy(), X_test.copy()
        if as_category:
            for c in CAT:
                cats = sorted(set(X_train[c]) | set(X_test[c]))
                for X in (Xa, Xb, Xq):
                    X[c] = pd.Categorical(X[c], categories=cats)
        m = make(); m.fit(Xa, y[tr_idx])
        oof[va_idx] = m.predict(Xb)
        pred_test += m.predict(Xq) / gkf.get_n_splits()
    return oof, pred_test


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


print("\nセットアップ完了。ヘルパー: check / call_safely / rmse_log / make_oof")


## ① なぜ: 最終日にやることは「混ぜる」

コンペの終盤、上位陣がほぼ必ずやるのが**アンサンブル**だ。複数のモデルの予測を混ぜると、
単独より安定して良くなることが多い。理屈は単純で、モデルごとに間違え方が違うなら、
平均すると誤差が打ち消し合うからだ。

実務でも同じことをする。ただし実務では「混ぜると運用が2倍重くなる」という代償があるので、
**本当に混ぜる価値があるのか**を数字で確かめてから決める。今日はその判断のしかたを身につける。

## ② 解説: 重みは「手元のデータ」で決める

ブレンドは2つの予測を `w * A + (1 - w) * B` の形で混ぜるだけ。問題は **w をどう決めるか**。

| やり方 | 何が起きるか |
|---|---|
| LB のスコアを見て w を選ぶ | **LB に過学習する**。public LB の300件に合わせただけになり、private で崩れる |
| **OOF 予測で w を選ぶ** | 手元のデータだけで決まる。**これが正解** |
| 何も考えず 0.5 ずつ | 運が良ければ効く。悪ければ良い方の足を引っ張る(② の後半で実測する) |

OOF 予測(out-of-fold)は Day2 で作ったもの — 「**その行を学習に使っていないモデルによる予測**」を
全行ぶん集めた配列だ。これがあるから、test に一度も触れずに w を決められる。

今日は2つのモデルを使う。どちらも Day3・Day4 で作ったものと同じだ。

| モデル | 中身 | 由来 |
|---|---|---|
| `ridge_model()` | 欠損補完 → 標準化 → one-hot → Ridge | Day4 の Pipeline |
| `lgbm_model()` | LightGBM(カテゴリは `category` dtype) | Day3 の主砲 |

In [ ]:
# GOAL: 2つのモデルの OOF を作り、単独スコアを比べる

# STEP 1: OOF を作る(Day2 で確定した GroupKFold(product_key) を使う)
t0 = time.time()
oof_ridge, pt_ridge = make_oof(ridge_model, as_category=False)
oof_lgbm, pt_lgbm = make_oof(lambda: lgbm_model(0), as_category=True)
print(f"OOF 作成: {time.time() - t0:.1f} 秒")

# STEP 2: OOF の形を確認する(長さが train と同じ = 全行に予測が1つずつ付いている)
print("oof_ridge:", oof_ridge.shape, " oof_lgbm:", oof_lgbm.shape, " y:", y.shape)

# STEP 3: 単独スコア
print(f"\n{'モデル':22s} {'OOF(手元)':>10s}")
print(f"{'Ridge(Pipeline)':22s} {rmse_log(oof_ridge, y):10.4f}")
print(f"{'LightGBM':22s} {rmse_log(oof_lgbm, y):10.4f}")
print("\n→ このコンペでは Ridge の方が強い。Day3 で主砲と呼んだ LightGBM が負けている。")

## ④ 予測: 混ぜたら、どのくらい上がる?

次のセルで、重み `w` を 0 から 1 まで 0.05 刻みで振って、OOF スコアがどう動くかを見る。
`w` は LightGBM 側の重みで、`w=0` なら Ridge 100%、`w=1` なら LightGBM 100% だ。

実行する前に予測してほしい。

1. スコアが最小になる `w` は、だいたいどのあたりだと思う?
2. その最小値は、良い方の単独スコア(0.6378)からどのくらい良くなると思う?
3. もし「最小になる `w` が 0 ちょうど」だったら、それは何を意味する?

In [ ]:
# GOAL: 重みを振って、OOF スコアの曲線を見る

w_grid = np.round(np.arange(0, 1.0001, 0.05), 2)
scores = [rmse_log(w * oof_lgbm + (1 - w) * oof_ridge, y) for w in w_grid]

print(f"{'w(LightGBM側)':>14s} {'OOF':>9s}")
for w, s in zip(w_grid, scores):
    mark = "  ← 最小" if s == min(scores) else ""
    if round(w * 100) % 10 == 0:          # 見やすさのため 0.1 刻みで表示
        print(f"{w:14.2f} {s:9.4f}{mark}")

best_i = int(np.argmin(scores))
print(f"\n最適 w = {w_grid[best_i]:.2f}  →  OOF {scores[best_i]:.4f}")
print(f"良い方の単独(Ridge) = {rmse_log(oof_ridge, y):.4f}")
print(f"改善幅 = {rmse_log(oof_ridge, y) - scores[best_i]:+.4f}")
print("\n→ w = 0.00。つまり『LightGBM を混ぜない』のが最善という結論になった。")
print("  混ぜる価値はゼロ。この事実から目を逸らさずに、次で原因を突き止める。")

## ⑥ 書いてみる: 最適な重みを返す関数

重み探索は毎回やるので関数にしておく。演習 `ex01` でもそのまま部品になる。

次のセルで `blend_search(oof_a, oof_b, y_true, grid)` を書こう。

- `grid` の各 `w` について `w * oof_a + (1 - w) * oof_b` のスコアを `rmse_log` で測る
- **スコアが最小になる `w`** と、そのときのスコアを `(best_w, best_score)` のタプルで返す
- `best_w` は Python の `float`、`best_score` も `float`

3〜5行で書ける。`min(候補, key=...)` を使ってもいいし、リストを作って `np.argmin` でもよい。

> ここで測っているのは `oof_a` 側の重みだ。呼び出すときの引数の順番に注意すること。

In [ ]:
def blend_search(oof_a, oof_b, y_true, grid):
    """oof_a の重み w を grid から探し、(best_w, best_score) を返す。"""
    # ここに書く(ヒント: 各 w について rmse_log(w * oof_a + (1 - w) * oof_b, y_true) を測り、
    #                    最小になった w とスコアを返す)
    return None


result = call_safely(blend_search, oof_lgbm, oof_ridge, y, w_grid)
print("blend_search(LightGBM, Ridge) =", result)

In [ ]:
# ===== チェックポイント 1: ブレンド重みの探索 =====
_r = result if isinstance(result, tuple) and len(result) == 2 else (None, None)

check("A-1 最適な w", _r[0], 0.0,
      hint="LightGBM 側の重み。混ぜない方が良いので 0.0 になる。")
check("A-2 そのときのスコア", None if _r[1] is None else round(_r[1], 4), 0.6378,
      hint="w=0 なら Ridge 単独と同じ値。rmse_log を使うこと。")
check("A-3 返り値は2つ組か", None if result is None else len(result), 2,
      hint="(best_w, best_score) のタプルを返す。")
check("A-4 best_w は float か", None if _r[0] is None else float(isinstance(_r[0], float)), 1.0,
      hint="np.float64 なら float(...) で包む。")

print("\n(4つとも [OK] になったら次の概念へ)")

## ① なぜ: 「効かなかった」で終わらせない

混ぜても効かない、は実務で頻繁に起きる。ここで「アンサンブルはこのタスクに効かないんだな」で
終わらせると、次も同じ場所で時間を溶かす。**なぜ効かないのかを数字で言えるようになる**のが今日の山場だ。

そして原因が分かれば、「どういうモデルを足せば効くのか」も分かる。

## ② 解説: 予測どうしの相関を見る

アンサンブルが効く条件はひとつだけ。**モデルたちが違う間違え方をしていること**。

同じ間違え方をしているなら、平均しても間違いは消えない。逆に、片方が外すところで
もう片方が当たっているなら、平均すると誤差が打ち消し合う。

これは **OOF 予測どうしの相関**で測れる。

| 相関 | 意味 |
|---|---|
| 0.99 以上 | ほぼ同じことを言っている。混ぜても無駄 |
| 0.90 前後 | かなり似ている。効いても小さい |
| 0.7 以下 | 違う情報を持っている。混ぜる価値がある |

C# で言えば、2つの `IPredictor` 実装が**同じロジックのコピー**なのか、
**別のアルゴリズムを実装したもの**なのか、という違いだ。インターフェースが同じでも中身が同じなら、
2つ持つ意味はない。

In [ ]:
# GOAL: 相関を測って「効かない理由」を確かめ、効く場合と並べて比べる

# STEP 1: このコンペの2モデルの OOF 相関
corr_real = float(np.corrcoef(oof_ridge, oof_lgbm)[0, 1])
print(f"Ridge と LightGBM の OOF 相関: {corr_real:.4f}  ← 0.94。ほぼ同じことを言っている")

# STEP 2: なぜここまで似るのか。このデータの成り立ちを思い出す。
print("""
このコンペのデータは、対数空間で『カテゴリ水準 + 状態 + サイト係数 + 時間トレンド + ノイズ』
という足し算で作られている。つまり **Ridge が真の生成モデルそのもの**だ。
LightGBM はその同じ関数を木で近似しているだけなので、原理的に勝てないし、
持っている情報も Ridge の部分集合になる。だから混ぜても足せるものが無い。""")

# STEP 3: では「効く場合」はどう見えるのか。意図的に作った例で確かめる。
#   y = 3*x1 + 2*sin(2.5*x2) + ノイズ  という、線形部分と非線形部分を持つデータ。
#   モデルA は x1 だけ、モデルB は x2 だけを見る = 見ている情報が重ならない。
rng = np.random.default_rng(0)
N = 1500
x1 = rng.normal(0, 1, N)
x2 = rng.uniform(-3, 3, N)
y_demo = 3.0 * x1 + 2.0 * np.sin(2.5 * x2) + rng.normal(0, 0.5, N)

def demo_oof(make, X):
    o = np.zeros(N)
    for a, b in KFold(5, shuffle=True, random_state=0).split(X):
        m = make(); m.fit(X[a], y_demo[a]); o[b] = m.predict(X[b])
    return o

oof_a = demo_oof(lambda: Ridge(alpha=1.0), x1.reshape(-1, 1))
oof_b = demo_oof(lambda: lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1, n_jobs=1),
                 x2.reshape(-1, 1))

print(f"\n【構成した例】")
print(f"  モデルA: Ridge(x1のみ)      OOF RMSE {rmse_log(oof_a, y_demo):.4f}")
print(f"  モデルB: LightGBM(x2のみ)   OOF RMSE {rmse_log(oof_b, y_demo):.4f}")
print(f"  OOF 相関: {float(np.corrcoef(oof_a, oof_b)[0, 1]):+.4f}  ← ほぼ 0。見ている情報が重ならない")

## ④ 予測: 単純平均(50/50)は、良い方の単独より良い?

構成した例で、モデルA は 1.4695、モデルB は 3.1802 だった。
B は A の倍以上悪い。この2つを混ぜる。

実行する前に予測してほしい。

1. **重みを最適化した**ブレンドは、良い方の単独(1.4695)より良くなると思う?
2. **何も考えず 50/50 で平均**したら、どうなると思う? 良い方より良い? 悪い?
3. そもそも「x1 と x2 の両方を1つのモデルに入れる」ことができるなら、それと比べてどうだろう?

In [ ]:
# GOAL: 相関の低いモデルを混ぜると何が起きるか、3通り並べて確かめる

w_grid = np.round(np.arange(0, 1.0001, 0.05), 2)
d_scores = [rmse_log(w * oof_a + (1 - w) * oof_b, y_demo) for w in w_grid]
d_best_i = int(np.argmin(d_scores))

print(f"{'やり方':34s} {'OOF RMSE':>10s}")
print(f"{'モデルA 単独':34s} {rmse_log(oof_a, y_demo):10.4f}")
print(f"{'モデルB 単独':34s} {rmse_log(oof_b, y_demo):10.4f}")
print(f"{'単純平均(50/50)':34s} {rmse_log(0.5 * oof_a + 0.5 * oof_b, y_demo):10.4f}   ← 良い方より悪い!")
print(f"{f'最適ブレンド(w={w_grid[d_best_i]:.2f})':34s} {d_scores[d_best_i]:10.4f}   ← 良い方より改善")

# 参考: そもそも1つのモデルに両方の特徴を入れられるなら
oof_both = demo_oof(lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, verbose=-1, n_jobs=1),
                    np.c_[x1, x2])
print(f"{'1つのモデルに x1,x2 両方を入れる':34s} {rmse_log(oof_both, y_demo):10.4f}   ← 圧勝")

print("""
→ 3つのことが分かる。
   1. 相関が低ければブレンドは効く(良い方の単独より改善した)
   2. だが **単純平均は良い方の足を引っ張る**。重みは必ず OOF で学習すること
   3. そして **1つのモデルに全部の特徴を入れられるなら、その方がずっと強い**。
      アンサンブルは『特徴量を1つのモデルにまとめられないとき』の最後の手段であって、
      最初に手を出すものではない。""")

## ⑥ 書いてみる: 相関から「混ぜる価値」を判定する

実務では毎回この判定をする。関数にしておこう。

次のセルで2つ作る。

| 変数 | 中身 |
|---|---|
| `pred_corr(a, b)` | 2つの予測配列の**ピアソン相関係数**を `float` で返す関数 |
| `worth_blending(a, b, threshold=0.95)` | 相関が `threshold` **未満**なら `True`(混ぜる価値あり)、以上なら `False` を返す関数 |

使う道具: `np.corrcoef(a, b)` は **2x2 の行列**を返す。欲しいのは `[0, 1]` の要素。

**2〜4行**で書ける。`worth_blending` は `pred_corr` を呼べばよい。

In [ ]:
def pred_corr(a, b):
    """2つの予測のピアソン相関係数を float で返す。"""
    # ここに書く(ヒント: np.corrcoef は 2x2 行列を返す。欲しいのは [0, 1] の要素)
    return None


def worth_blending(a, b, threshold=0.95):
    """相関が threshold 未満なら True(混ぜる価値がある)。"""
    # ここに書く(ヒント: pred_corr を呼んで比べるだけ)
    return None


print("このコンペ  相関:", call_safely(pred_corr, oof_ridge, oof_lgbm),
      " 混ぜる価値:", call_safely(worth_blending, oof_ridge, oof_lgbm))
print("構成した例  相関:", call_safely(pred_corr, oof_a, oof_b),
      " 混ぜる価値:", call_safely(worth_blending, oof_a, oof_b))

In [ ]:
# ===== チェックポイント 2: 相関で混ぜる価値を判定する =====
_corr_real = call_safely(pred_corr, oof_ridge, oof_lgbm)
_corr_demo = call_safely(pred_corr, oof_a, oof_b)
_wb_real = call_safely(worth_blending, oof_ridge, oof_lgbm)

check("B-1 このコンペの OOF 相関", None if _corr_real is None else round(_corr_real, 4), 0.9392,
      hint="np.corrcoef(a, b)[0, 1]。0.94 前後になる。")
check("B-2 構成した例の OOF 相関", None if _corr_demo is None else round(_corr_demo, 4), -0.0336,
      hint="ほぼ 0(わずかに負)。見ている情報が重ならないため。")
check("B-3 このコンペは混ぜる価値があるか", None if _wb_real is None else float(bool(_wb_real)), 0.0,
      hint="0.94 は既定の閾値 0.95 未満なので True になってしまう。閾値は目安であって、"
           "最後は実際に重み探索して確かめること。ここでは threshold=0.90 で判定してみよう。")
check("B-4 構成した例は混ぜる価値があるか", float(bool(call_safely(worth_blending, oof_a, oof_b))), 1.0,
      hint="相関がほぼ 0 なので True。")

print("\n(4つとも [OK] になったら次の概念へ)")

## ① なぜ: 同じ設定なのにスコアが動く

「昨日と同じコードを回したのにスコアが変わった」は誰もが一度は踏む。原因は乱数だ。
GBDT は行や列をランダムに間引きながら木を育てるので、シードが違えば別のモデルになる。

この揺れを**平均で潰す**のがシード平均(seed averaging)。安上がりで確実に効く、
コンペ終盤の定番テクニックだ。ただし**効かない場合がある**。そこが今日の勘所。

## ② 解説: 乱数が入っていなければ、平均しても意味がない

LightGBM の乱数性は主にこの2つのパラメータから来る。

| パラメータ | 何をするか |
|---|---|
| `subsample`(+ `subsample_freq`) | 木を1本作るたびに、**行**をランダムに間引く |
| `colsample_bytree` | 木を1本作るたびに、**列**をランダムに間引く |

**どちらも既定では 1.0**、つまり間引かない。この状態では `random_state` を変えても
**まったく同じ木**が育つ。シード平均は完全に無駄になる。

まずそれを実演してから、乱数を入れた場合を測る。

In [ ]:
# GOAL: 乱数が無いとシードを変えても同じ、を確かめてから、シード平均の効果を測る

# STEP 1: bagging なし(既定)で seed だけ変える
t0 = time.time()
no_bag = [make_oof(lambda s=s: lgbm_model(s, bagging=False), True)[0] for s in (0, 1, 2)]
same = np.allclose(no_bag[0], no_bag[1]) and np.allclose(no_bag[1], no_bag[2])
print(f"bagging なしで seed 0/1/2 の OOF は完全一致するか: {same}")
print(f"  seed0 {rmse_log(no_bag[0], y):.4f} / seed1 {rmse_log(no_bag[1], y):.4f} / seed2 {rmse_log(no_bag[2], y):.4f}")
print("  → 一致する。この状態でシード平均をしても、同じものを5回足して5で割るだけ。")

# STEP 2: bagging あり(subsample=0.8, colsample_bytree=0.8)で5シード
bags = [make_oof(lambda s=s: lgbm_model(s, bagging=True), True) for s in range(5)]
each = [rmse_log(o, y) for o, _ in bags]
print(f"\nbagging ありの各シードの OOF: {[round(s, 4) for s in each]}")
print(f"  ばらつき(標準偏差): {np.std(each):.5f}")

avg_oof = np.mean([o for o, _ in bags], axis=0)
print(f"\n{'':28s} {'OOF':>9s}")
print(f"{'単一シード(seed=0)':28s} {each[0]:9.4f}")
print(f"{'5シード平均':28s} {rmse_log(avg_oof, y):9.4f}")
print(f"改善 {each[0] - rmse_log(avg_oof, y):+.4f}   ({time.time() - t0:.0f} 秒)")

## ④ 予測: シード平均は「予測を平均」する? 「スコアを平均」する?

上のセルでは、5つのシードの**予測配列を平均**してから1回スコアを測った。
もうひとつのやり方として、**5つのスコアを平均する**こともできる。

実行する前に考えてほしい。

1. この2つは同じ値になると思う? 違うとしたら、どちらが小さくなる?
2. 本番の submission を作るとき、あなたが提出するのはどちらの結果?
3. シードを 5 から 20 に増やしたら、改善は5倍になると思う?

In [ ]:
# GOAL: 「予測の平均」と「スコアの平均」は別物であることを確かめる

score_of_avg = rmse_log(avg_oof, y)          # 予測を平均してから測る
avg_of_score = float(np.mean(each))          # スコアを測ってから平均する

print(f"予測を平均してから測る : {score_of_avg:.4f}   ← これがシード平均の効果")
print(f"スコアを測ってから平均 : {avg_of_score:.4f}   ← ただの平均値。何も改善していない")
print(f"差: {avg_of_score - score_of_avg:+.4f}")
print("""
→ 予測を平均すると誤差が打ち消し合うので、スコアは各シードの平均より良くなる。
  これは数学的に必ずそうなる(凸性)。提出するのは『予測を平均したもの』。

→ シードを増やすほど改善するが、頭打ちになる。5→20 にしても効果は5倍にはならない。
  学習時間は素直に4倍になるので、**どこで止めるかはコストとの相談**になる。
  ここは概念5 のコスト設計に直結する。""")

## ⑥ 書いてみる: シード平均を関数にする

次のセルで `seed_average(seeds, bagging=True)` を書こう。

- 各 `seed` について `make_oof(lambda: lgbm_model(seed, bagging=bagging), True)` を呼ぶ
  (戻り値は `(oof, pred_test)` の2つ組)
- **OOF の平均**と **test 予測の平均**を作り、`(avg_oof, avg_pred_test)` のタプルで返す

**4〜6行**で書ける。`np.mean([...], axis=0)` は「配列のリストを要素ごとに平均する」。
`axis=0` を忘れると全部を1つの数にしてしまうので注意(既習の axis の話がここでも効く)。

> 時間がかかるので、チェックは `seeds=(0, 1)` の2つだけで行う。

In [ ]:
def seed_average(seeds, bagging=True):
    """複数シードの OOF と test 予測をそれぞれ平均して (avg_oof, avg_pred_test) を返す。"""
    # ここに書く(ヒント: 各 seed で make_oof(lambda: lgbm_model(seed, bagging=bagging), True) を呼び、
    #                    oof のリストと pred_test のリストをそれぞれ np.mean(..., axis=0) する)
    return None


sa = call_safely(seed_average, (0, 1))
print("seed_average((0, 1)) =", None if sa is None else (np.asarray(sa[0]).shape, np.asarray(sa[1]).shape))

In [ ]:
# ===== チェックポイント 3: シード平均 =====
_sa = sa if isinstance(sa, tuple) and len(sa) == 2 else (None, None)

check("C-1 OOF の長さ", None if _sa[0] is None else len(_sa[0]), len(y),
      hint="train の行数と同じ。axis=0 を忘れるとスカラーになる。")
check("C-2 test 予測の長さ", None if _sa[1] is None else len(_sa[1]), len(X_test),
      hint="test の行数と同じ。")
check("C-3 2シード平均の OOF スコア",
      None if _sa[0] is None else round(rmse_log(_sa[0], y), 4),
      round(rmse_log(np.mean([bags[0][0], bags[1][0]], axis=0), y), 4),
      hint="bagging=True(既定)で seed 0 と 1 を平均する。")
check("C-4 bagging なしでは改善しないこと",
      float(np.allclose(no_bag[0], no_bag[1])), 1.0,
      hint="このチェックは実演の再確認。bagging なしなら seed を変えても同じ。")

print("\n(4つとも [OK] になったら次の概念へ)")

## ① なぜ: 学習と推論は、別の場所で動く

君の構成はこうだった。

| 役割 | 場所 | 頻度 |
|---|---|---|
| 学習 | クラウドGPU(時間借り) | 月に数回・数時間 |
| 推論 | CPU(常時稼働) | 毎日・大量 |

**別のマシンで、別の日に動く**。だから「学習して、そのまま予測する」1本のスクリプトでは運用できない。
学習の成果物を**ファイルに固めて持ち運ぶ**必要がある。これを artifact と呼ぶ。

ここを雑にやると、「本番の予測が学習時と微妙に違う」という最も追いにくいバグを踏む。

## ② 解説: 前処理ごと1つのオブジェクトに固める

artifact に入れるべきものは**モデルの重みだけではない**。

| 入れるもの | 入れないと何が起きるか |
|---|---|
| 前処理込みの推定器(`Pipeline` 全体) | 推論側で前処理を書き直すことになり、必ずズレる |
| 特徴量の列名と順序 | 列の順番が変わって静かに壊れる |
| 学習に使ったライブラリのバージョン | 挙動が変わったときに原因を特定できない |
| seed などの再現に必要な設定 | 「同じ結果が出ない」を追えなくなる |

Day4 で `Pipeline` を「前処理を推定器の一部にする道具」として学んだのが、ここで効いてくる。
**前処理が推定器の中に入っているから、1ファイル保存するだけで推論側が完結する**。

C# で言えば、DI コンテナに組み上げたオブジェクトグラフをまるごとシリアライズして、
別プロセスで復元するようなものだ。保存するのは「重み」ではなく「組み上がった部品」。

道具は `joblib`。`joblib.dump(obj, path)` / `joblib.load(path)` の2つだけ。

In [ ]:
# GOAL: artifact を保存して読み直し、同じ予測が出ることを確かめる

# STEP 1: 全 train で学習し直す(検証は済んだので、最終モデルは手元の全データを使う)
final_model = ridge_model()
final_model.fit(X_train, y)

# STEP 2: モデル本体だけでなく、再現に必要な情報を一緒に固める
artifact = {
    "model": final_model,
    "feature_columns": list(X_train.columns),   # 列の順序も含めて保存する
    "origin_date": str(ORIGIN.date()),          # days の起点。ズレると全予測がずれる
    "target": "log1p(price)",                   # 予測値が何なのかを明記する
    "sklearn_version": sklearn.__version__,
    "created_at": "2026-08-09",
}
path = OUT / "model_artifact.joblib"
joblib.dump(artifact, path)
print("保存:", path, f"({path.stat().st_size / 1024:.0f} KB)")

# STEP 3: 別プロセスのつもりで読み直す
loaded = joblib.load(path)
print("読み込んだキー:", sorted(loaded.keys()))
print("列の順序が一致:", loaded["feature_columns"] == list(X_test.columns))

# STEP 4: 同じ予測が出るか
pred_before = final_model.predict(X_test)
pred_after = loaded["model"].predict(X_test[loaded["feature_columns"]])
print(f"\n予測が完全一致: {np.array_equal(pred_before, pred_after)}")
print(f"最大の差: {np.max(np.abs(pred_before - pred_after)):.2e}")

## ④ 予測: 列の順序を入れ替えたら、どうなる?

上では `X_test[loaded["feature_columns"]]` と、**保存した順序で列を並べ直してから**予測した。
もし並べ直さずに、順序の違う DataFrame を渡したらどうなるだろう。

実行する前に予測してほしい。

1. 例外が出て止まると思う? それとも黙って動くと思う?
2. 黙って動く場合、予測値は正しいと思う?
3. この違いは `Pipeline`(列名で選ぶ)と、NumPy 配列(位置で選ぶ)でどう変わるだろう?

In [ ]:
# GOAL: 列の順序が狂ったときに何が起きるかを見る

shuffled = X_test[list(reversed(list(X_test.columns)))]   # 列を逆順にしただけ
print("元の列  :", list(X_test.columns))
print("逆順の列:", list(shuffled.columns))

try:
    pred_shuffled = loaded["model"].predict(shuffled)
    print(f"\n例外は出なかった。最大の差: {np.max(np.abs(pred_before - pred_shuffled)):.2e}")
    print("→ ColumnTransformer は**列名**で選ぶので、順序が違っても正しく動く。")
except Exception as e:
    print(f"\n例外: {type(e).__name__}: {e}")

# ただし NumPy 配列に変換して渡すと、位置で解釈されるので静かに壊れる
print("""
→ ただしこれは Pipeline が列名で選んでいるから助かっているだけだ。
  `.to_numpy()` して渡すと位置で解釈されるので、**例外も出ずに間違った予測**が返る。
  だから artifact に feature_columns を保存し、**推論側で必ず並べ直す**のが型になる。
  『動いているから正しい』は成り立たない。""")

## ⑥ 書いてみる: 保存と推論を関数に分ける

学習側と推論側を、**別々の関数**にする。これがそのまま2つのスクリプトに分かれることになる。

次のセルで2つ作る。

| 関数 | 中身 |
|---|---|
| `save_artifact(model, columns, path)` | `{"model": model, "feature_columns": list(columns)}` を `joblib.dump` して、`path` を返す |
| `load_and_predict(path, df)` | `joblib.load` して、`df` を**保存された列順に並べ直してから** `predict` した結果を返す |

**それぞれ2〜3行**で書ける。`load_and_predict` で並べ直すのを忘れないこと
(上で見たとおり、忘れても動いてしまうことがあるから怖い)。

In [ ]:
def save_artifact(model, columns, path):
    """model と列順を1つの dict にまとめて保存し、path を返す。"""
    # ここに書く(ヒント: joblib.dump({"model": ..., "feature_columns": list(columns)}, path))
    return None


def load_and_predict(path, df):
    """保存した artifact を読み、df を保存時の列順に並べ直してから predict する。"""
    # ここに書く(ヒント: a = joblib.load(path) のあと a["model"].predict(df[a["feature_columns"]]))
    return None


_p = call_safely(save_artifact, final_model, X_train.columns, OUT / "my_artifact.joblib")
_pred = call_safely(load_and_predict, _p, X_test) if _p is not None else None
print("保存先:", _p)
print("予測の形:", None if _pred is None else np.asarray(_pred).shape)

In [ ]:
# ===== チェックポイント 4: artifact による学習と推論の分離 =====
check("D-1 予測の長さ", None if _pred is None else len(_pred), len(X_test),
      hint="test の行数と同じ。")
check("D-2 元の予測と完全一致するか",
      None if _pred is None else float(np.allclose(np.asarray(_pred), pred_before)), 1.0,
      hint="同じモデル・同じ入力なので一致するはず。列を並べ直したか確認する。")
check("D-3 ファイルが実際に作られたか",
      0.0 if _p is None else float(Path(_p).exists()), 1.0,
      hint="joblib.dump したあと path を return すること。")
check("D-4 列順を逆にしても同じ結果になるか",
      None if _p is None else float(np.allclose(
          np.asarray(call_safely(load_and_predict, _p, X_test[list(reversed(list(X_test.columns)))])),
          pred_before)), 1.0,
      hint="load_and_predict の中で df[a['feature_columns']] と並べ直していれば一致する。")

print("\n(4つとも [OK] になったら次の概念へ)")

## ① なぜ: 「LLM API より圧倒的に安い」を数字で示す

自前でモデルを持つ理由はひとつ、**そのほうが安いから**だ。精度が同じなら、
運用が楽な API を使えばいい。だから「自前のほうが安い」を数字で言えないなら、
そもそも自前でやる理由がない。

上司に予算を通すときに出すのはこの計算だ。今日の締めくくりとして、自分で組み立てる。

## ② 解説: 学習と推論で、コストの形が違う

君の構成のコストは、**性質の違う2つ**に分かれる。

```
自前の月額コスト
  = 学習コスト(GPUインスタンス時間単価 × 1回あたり学習時間 × 月の再学習回数)
  + 推論コスト(CPUインスタンス時間単価 × 24時間 × 30日)
```

学習は**回数が少なく単価が高い**。推論は**単価が安いが常時動く**。だから件数が増えても
自前のコストは**ほとんど増えない**(CPU が捌ける限り)。

一方 API は:

```
API の月額コスト = 件数 × (入力トークン + 出力トークン) × トークン単価
```

**件数に正比例する**。ここが決定的な違いで、件数が増えるほど自前が有利になる。
両者が等しくなる件数が**損益分岐点**だ。

> 分類なら出力は1ラベルなので出力トークンはほぼゼロだが、**要約や生成は出力トークンが効く**
> (Day11 で扱う)。同じ件数でも生成タスクのほうが API は高くつく。

In [ ]:
# GOAL: 具体的な数字で損益分岐点を出す

# 想定(実際の見積もりに合わせて差し替えればよい)
GPU_HOURLY = 1.2       # GPUインスタンスの時間単価(ドル)
TRAIN_HOURS = 3.0      # 1回の学習にかかる時間
RETRAIN_PER_MONTH = 4  # 月の再学習回数
CPU_HOURLY = 0.10      # 推論用CPUインスタンスの時間単価(ドル)
HOURS_PER_MONTH = 24 * 30

TOKENS_IN = 400        # 1件あたりの入力トークン
TOKENS_OUT = 10        # 1件あたりの出力トークン(分類なのでごく短い)
PRICE_PER_1K = 0.002   # 1000トークンあたりの単価(ドル)

def self_cost(_items):
    """自前の月額。件数にほとんど依存しない。"""
    train = GPU_HOURLY * TRAIN_HOURS * RETRAIN_PER_MONTH
    infer = CPU_HOURLY * HOURS_PER_MONTH
    return train + infer

def api_cost(items):
    """APIの月額。件数に正比例する。"""
    return items * (TOKENS_IN + TOKENS_OUT) / 1000 * PRICE_PER_1K

print(f"{'月間件数':>12s} {'自前($)':>10s} {'API($)':>10s}  判定")
for items in (1_000, 10_000, 50_000, 100_000, 300_000, 1_000_000):
    s, a = self_cost(items), api_cost(items)
    print(f"{items:12,d} {s:10.2f} {a:10.2f}  {'自前が安い' if s < a else 'APIが安い'}")

print(f"\n自前の内訳: 学習 {GPU_HOURLY * TRAIN_HOURS * RETRAIN_PER_MONTH:.2f} + "
      f"推論 {CPU_HOURLY * HOURS_PER_MONTH:.2f} = {self_cost(0):.2f} ドル/月(件数に依らない)")

## ④ 予測: 分岐点はどのあたり?

上の表を見て、次を予測してから確かめよう。

1. 損益分岐点(自前とAPIが同額になる件数)は、だいたい何件くらい?
2. **再学習の頻度を月4回から月1回に減らす**と、分岐点はどちらに動く?
3. **要約タスク**にして出力トークンを 10 → 300 に増やすと、分岐点はどう動く?
4. 逆に、自前が絶対に勝てない条件はどんなときだろう?

In [ ]:
# GOAL: 条件を変えて分岐点がどう動くかを見る

def break_even(tokens_out=TOKENS_OUT, retrain=RETRAIN_PER_MONTH):
    """自前とAPIが同額になる月間件数を返す。"""
    fixed = GPU_HOURLY * TRAIN_HOURS * retrain + CPU_HOURLY * HOURS_PER_MONTH
    per_item = (TOKENS_IN + tokens_out) / 1000 * PRICE_PER_1K
    return fixed / per_item

print(f"{'条件':40s} {'分岐点(件/月)':>16s}")
print(f"{'既定(分類・月4回再学習)':40s} {break_even():16,.0f}")
print(f"{'再学習を月1回に減らす':40s} {break_even(retrain=1):16,.0f}")
print(f"{'要約タスク(出力300トークン)':40s} {break_even(tokens_out=300):16,.0f}")
print(f"{'要約 + 月1回再学習':40s} {break_even(tokens_out=300, retrain=1):16,.0f}")

print("""
→ 出力が長いタスクほど分岐点は**下がる**(= 少ない件数で自前が有利になる)。
  要約や生成を大量に回すなら、自前を検討する価値は分類より大きい。

→ ただし自前が勝てない条件もある。件数が分岐点よりずっと少ない、
  精度要求が高くて小型モデルでは届かない、運用の人手が確保できない、
  タスクが頻繁に変わってその都度学習し直す必要がある — こういう場合は API が正解だ。
  **『自前が安い』は件数と要求次第の条件付きの結論**であって、常に正しいわけではない。""")

## ⑥ 書いてみる: 損益分岐点を返す関数

最後の課題。次のセルで `monthly_break_even(fixed_cost, tokens_per_item, price_per_1k)` を書こう。

- `fixed_cost` … 件数に依存しない自前の月額(ドル)
- `tokens_per_item` … 1件あたりの合計トークン数(入力 + 出力)
- `price_per_1k` … 1000トークンあたりの単価(ドル)
- 返すのは**自前とAPIが同額になる月間件数**(`float`)

式は `fixed_cost / (tokens_per_item / 1000 * price_per_1k)`。**1行**で書ける。

> こういう1行の関数こそ関数にしておく価値がある。上司に説明するとき、
> 数字を差し替えて再計算できる形になっているかどうかで、話の進み方が変わる。

In [ ]:
def monthly_break_even(fixed_cost, tokens_per_item, price_per_1k):
    """自前とAPIが同額になる月間件数を返す。"""
    # ここに書く(ヒント: 1件あたりのAPI単価は tokens_per_item / 1000 * price_per_1k)
    return None


print("既定条件:", call_safely(monthly_break_even, self_cost(0), TOKENS_IN + TOKENS_OUT, PRICE_PER_1K))
print("要約条件:", call_safely(monthly_break_even, self_cost(0), TOKENS_IN + 300, PRICE_PER_1K))

In [ ]:
# ===== チェックポイント 5: コスト設計 =====
_fixed = self_cost(0)
check("E-1 既定条件の分岐点", call_safely(monthly_break_even, _fixed, TOKENS_IN + TOKENS_OUT, PRICE_PER_1K),
      round(_fixed / ((TOKENS_IN + TOKENS_OUT) / 1000 * PRICE_PER_1K), 6),
      hint="fixed_cost / (tokens_per_item / 1000 * price_per_1k)。")
check("E-2 要約条件(出力300)の分岐点", call_safely(monthly_break_even, _fixed, TOKENS_IN + 300, PRICE_PER_1K),
      round(_fixed / ((TOKENS_IN + 300) / 1000 * PRICE_PER_1K), 6),
      hint="トークンが増えると分岐点は下がる。")
check("E-3 単価が2倍になったら分岐点は半分か",
      call_safely(monthly_break_even, _fixed, TOKENS_IN + TOKENS_OUT, PRICE_PER_1K * 2),
      round(_fixed / ((TOKENS_IN + TOKENS_OUT) / 1000 * PRICE_PER_1K * 2), 6),
      hint="反比例する。")
check("E-4 返り値は float か",
      float(isinstance(call_safely(monthly_break_even, _fixed, 410, PRICE_PER_1K), float)), 1.0,
      hint="割り算の結果なので自然に float になる。")

print("\n(4つとも [OK] になったら振り返りへ)")

## 振り返り

以下に1〜2文で書いてみよう(チューターがこの記述を学習ノートとスキル判定に使う)。

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

ヒント: 今日いちばん意外だったのは何だったか。「混ぜれば良くなる」と思っていたことが
そうではなかった、という体験を言葉にしておくと後で効く。

## まとめ

| 学んだこと | 要点 |
|---|---|
| OOF ブレンド | 重みは **OOF で決める**。LB で決めるのは LB への過学習 |
| 効かない理由の特定 | **OOF 予測の相関**を見る。0.94 なら同じことを言っている |
| 効くのはどんなときか | 相関が低いとき。ただし**単純平均は良い方の足を引っ張る** |
| もっと大事なこと | **1つのモデルに全特徴を入れられるなら、その方が強い**。アンサンブルは最後の手段 |
| シード平均 | `subsample` が無いと**シードを変えても同じ木**。乱数が無ければ意味がない |
| 予測の平均 vs スコアの平均 | 提出するのは**予測を平均したもの**。スコアの平均は何も改善しない |
| artifact | `Pipeline` ごと保存する。**列順も一緒に保存し、推論側で並べ直す** |
| コスト設計 | 自前は件数に依らずほぼ定額、API は件数に正比例。**分岐点を計算して選ぶ** |

### 今日の実測値(このコンペで)

| | OOF |
|---|---|
| Ridge(Pipeline) | 0.6378 |
| LightGBM | 0.7249 |
| 最適ブレンド(w=0.00) | 0.6378 ← **改善なし** |
| 5シード平均 | 0.7166 |

### この先どこで使うか

- **Day11(文章要約)** — 生成タスクは出力トークン数がそのままコストに乗る。
  今日の損益分岐の式を、出力の長いタスクに当てはめ直すことになる。
- **実務** — artifact の作り方と「列順を保存する」習慣は、そのまま本番の推論サービスに載る。
  コスト試算は、上司に自前運用を提案するときの材料になる。

演習(`ex01`〜`ex04`)へ進もう。lesson を見ながらで OK。